In [1]:
import numpy as np
from tqdm import tqdm
import scipy.sparse as sp
import scanpy as sc

def read_adata(path):
    adata = sc.read_h5ad(path)
    return adata
adata = read_adata('./only_hvg/PBMC_only_hvg.h5ad')


In [ ]:
# analyze the distribution of libery size of exclude high
import numpy as np
import matplotlib.pyplot as plt

# 获取 library size 数据
# library_sizes = np.array(adata.obs['n_genes'])
# analyze the distribution of libery size of exclude high

In [2]:
# 按cartessian_key划分细胞
import numpy as np
from tqdm import tqdm
import scipy.sparse as sp
import scanpy as sc

def read_adata(path):
    adata = sc.read_h5ad(path)
    return adata

def iter_adata_rows(
    adata,
    cols=("donor", "cell_type", "cytokine"),
    to_float32: bool = False,
):

    # 1) obs三列一次性取出，避免每行反复做 pandas 索引
    obs_df = adata.obs.loc[:, list(cols)]

    # 2) X：若是稀疏，尽量保证 CSR 以便按行切片更快
    # X = sp.csr_matrix(adata.X)
    
    X = sp.csr_matrix(adata.X)

    is_sparse = sp.issparse(X)

    for i in tqdm(range(adata.n_obs)):
        row_obs = obs_df.iloc[i]

        # 取obs字段
        out = {"i": i}
        for c in cols:
            out[c] = row_obs[c]

        out["x"] = X[i]
        yield out

group_dict = {}
# 用法示例

for item in iter_adata_rows(adata):
    donor, cell_type, cytokine, x = item["donor"], item["cell_type"], item["cytokine"], item["x"]
    cell_key = (donor, cell_type, cytokine)
    if cell_key not in group_dict:
        group_dict[cell_key] = []
    group_dict[(donor, cell_type, cytokine)].append(x)

100%|██████████| 9697974/9697974 [12:08<00:00, 13309.09it/s]


In [4]:
keys = list(group_dict.keys())
for k in tqdm(keys):
    if k[-1]!="PBS" and len(group_dict[k]) <= 3:
        group_dict.pop(k, None)

100%|██████████| 19458/19458 [00:00<00:00, 619879.90it/s]


In [5]:
import pickle

# processed_cartesian_keys 已在上一个 cell 中按 min_cells 过滤（默认保留 n_cells >= 3）
# 为了防止单独运行本 cell，这里做一个兜底：如果不存在就退化为不过滤
if "processed_cartesian_keys" not in globals():
    processed_cartesian_keys = list(group_dict.keys())

# 按1024 shard_size 进行配对的并保存
control_name = "PBS"
shard_size = 1024
rng = np.random.default_rng(0)
pert_sample_list = []
control_dict = {}

def pad_for_shard_size(array, shard_size):
    N = array.shape[0]
    pad_n = (-N) % shard_size  # number of rows to add
    if pad_n == 0:
        return array, np.empty((0,), dtype=np.int64)
    pad_idx = rng.integers(0, N, size=pad_n)   # with replacement
    array_pad = sp.vstack([array, array[pad_idx]])
    return array_pad, pad_idx

def equal_pert_parts(pert_array, shard_size=1024):
    
    pert_array_pad, _ = pad_for_shard_size(pert_array, shard_size)
    pert_num = pert_array_pad.shape[0]

    
    for sample_idx, i in enumerate(range(0, pert_num, shard_size)):
        pert_sub = pert_array_pad[i:i+shard_size]
        sample = {
            "cartesian_key": pck,
            "cell_matrix": pert_sub,
        }
        pert_sample_list.append(sample)


for pck in tqdm(processed_cartesian_keys):
    if pck[2] == control_name:
        
        control_dict[pck] = sp.vstack(group_dict[pck], format="csr")
    else:
        pert_array = sp.vstack(group_dict[pck], format="csr")
        equal_pert_parts(pert_array, shard_size=shard_size)
    

len(pert_sample_list)

with open(f"pert_sample_list.pkl", "wb") as f:
    pickle.dump(pert_sample_list, f)

with open(f"control_dict.pkl", "wb") as f:
    pickle.dump(control_dict, f)


100%|██████████| 18350/18350 [01:25<00:00, 215.65it/s]
